<a href="https://colab.research.google.com/github/SiyumiJayawardhane/freshsense-imagemodel/blob/siyumi/FreshSenseYoloV5ImageModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Define paths and create the new YOLO dataset structure**

In [ ]:
import os
import shutil
import random
from pathlib import Path

BASE_DRIVE_PATH = "/content/drive/MyDrive/FreshSense/Dataset"

# New unified dataset folder
DATASET_ROOT = "/content/drive/MyDrive/FreshSense/food_spoilage_dataset"
os.makedirs(f"{DATASET_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/images/val", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/labels/train", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/labels/val", exist_ok=True)

print("Folder structure created")

Folder structure created


**3. Reorganize + Remap class IDs**

In [ ]:
# Global 9-class list (order matters!)
class_names = [
    "fresh_banana", "atrisk_banana", "spoiled_banana",
    "fresh_cucumber", "atrisk_cucumber", "spoiled_cucumber",
    "fresh_tomato", "atrisk_tomato", "spoiled_tomato"
]

# Mapping: original class_id (per item) → global class_id
banana_mapping   = {0: 1, 1: 0, 2: 2}   # atrisk=0→1, fresh=1→0, spoiled=2→2
cucumber_mapping = {0: 4, 1: 3, 2: 5}
tomato_mapping   = {0: 7, 1: 6, 2: 8}

datasets = {
    "banana_dataset":   ("banana",   banana_mapping),
    "cucumber_dataset": ("cucumber", cucumber_mapping),
    "tomato_dataset":   ("tomato",   tomato_mapping)
}

all_image_label_pairs = []   # will store (image_path, label_path, new_class_id)

for dataset_name, (item_name, mapping) in datasets.items():
    dataset_path = os.path.join(BASE_DRIVE_PATH, dataset_name)
    if not os.path.exists(dataset_path):
        print(f" {dataset_name} not found")
        continue

    # The three class folders inside each dataset
    class_folders = ["atrisk", "fresh", "spoiled"] if item_name != "banana" else ["atrisk_banana", "fresh_banana", "spoiled_banana"]

    for idx, folder in enumerate(class_folders):
        class_dir = os.path.join(dataset_path, folder)
        if not os.path.exists(class_dir):
            print(f"Missing folder: {class_dir}")
            continue

        images_dir = os.path.join(class_dir, "images")
        labels_dir = os.path.join(class_dir, "labels")

        if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
            continue

        original_class_id = idx   # 0,1,2 based on folder order
        global_class_id = mapping[original_class_id]

        for img_file in os.listdir(images_dir):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            base_name = os.path.splitext(img_file)[0]
            label_file = base_name + ".txt"
            label_path = os.path.join(labels_dir, label_file)

            if os.path.exists(label_path):
                all_image_label_pairs.append((os.path.join(images_dir, img_file), label_path, global_class_id))

print(f"Found {len(all_image_label_pairs)} labeled images ready for merging")

Found 3201 labeled images ready for merging


**4. Copy files + Remap class IDs + Train/Val split (80/20)**

In [5]:
random.shuffle(all_image_label_pairs)
split_idx = int(len(all_image_label_pairs) * 0.8)

for i, (img_src, label_src, new_class) in enumerate(all_image_label_pairs):
    is_train = i < split_idx
    split = "train" if is_train else "val"

    # Copy image
    img_dest = f"{DATASET_ROOT}/images/{split}/{Path(img_src).name}"
    shutil.copy(img_src, img_dest)

    # Read label, change class_id, save
    with open(label_src, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if parts:
            parts[0] = str(new_class)          # ← remap class ID
            new_lines.append(" ".join(parts) + "\n")

    label_dest = f"{DATASET_ROOT}/labels/{split}/{Path(label_src).name}"
    with open(label_dest, "w") as f:
        f.writelines(new_lines)

print("Dataset successfully reorganized and split into train/val!")

Dataset successfully reorganized and split into train/val!


**5. Create data.yaml**

In [6]:
yaml_content = f"""train: {DATASET_ROOT}/images/train
val: {DATASET_ROOT}/images/val

nc: 9
names: {class_names}
"""

with open(f"{DATASET_ROOT}/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created")
!cat {DATASET_ROOT}/data.yaml

data.yaml created
train: /content/drive/MyDrive/FreshSense/food_spoilage_dataset/images/train
val: /content/drive/MyDrive/FreshSense/food_spoilage_dataset/images/val

nc: 9
names: ['fresh_banana', 'atrisk_banana', 'spoiled_banana', 'fresh_cucumber', 'atrisk_cucumber', 'spoiled_cucumber', 'fresh_tomato', 'atrisk_tomato', 'spoiled_tomato']


**6. Clone YOLOv5 and install requirements**

In [2]:
%cd /content
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt

/content
Cloning into 'yolov5'...
remote: Enumerating objects: 17889, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 17889 (delta 38), reused 10 (delta 10), pack-reused 17844 (from 5)
Receiving objects: 100% (17889/17889), 17.02 MiB | 21.31 MiB/s, done.
Resolving deltas: 100% (12179/12179), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 11.0 MB/s eta 0:00:00


**7. Train the model**

In [8]:
!python train.py --img 640 --batch 16 --epochs 100 --data {DATASET_ROOT}/data.yaml \
                 --weights yolov5s.pt --cache --project /content/FreshSense_Results \
                 --name yolov5s_food_spoilage

Streaming output truncated to the last 5000 lines.
      84/99      4.62G    0.02146    0.01378    0.00661         44        640:  68% 109/160 [00:30<00:12,  4.16it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      84/99      4.62G     0.0214    0.01376   0.006564         44        640:  69% 110/160 [00:30<00:11,  4.18it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      84/99      4.62G    0.02133    0.01376   0.006542         47        640:  69% 111/160 [00:30<00:12,  3.98it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      84/99      4.62G    0.021

# **Model Optimization**

**8. Model Prunning**

In [5]:
# 1. Pruning
!pip install torch-pruning -q
import torch
import torch.nn.utils.prune as prune

model_path = "/content/drive/MyDrive/FreshSense_Results/yolov5s_food_spoilage/weights/best.pt"
model = torch.load(model_path, map_location='cpu', weights_only=False)['model'].float()

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        prune.l1_unstructured(module, name='weight', amount=0.25)   # 25% pruning
        prune.remove(module, 'weight')

torch.save({'model': model}, "/content/drive/MyDrive/FreshSense_Results/yolov5s_food_spoilage/weights/best_pruned.pt")
print("Pruning completed (25% weights removed)")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Pruning completed (25% weights removed)


**9. Model export as a ONNX runtime**

In [7]:
# 2. Export pruned model to ONNX
!python /content/drive/MyDrive/FreshSense_yolov5_backup/export.py \
    --weights /content/drive/MyDrive/FreshSense_Results/yolov5s_food_spoilage/weights/best_pruned.pt \
    --include onnx \
    --imgsz 640 \
    --simplify

export: data=../drive/MyDrive/FreshSense_yolov5_backup/data/coco128.yaml, weights=['/content/drive/MyDrive/FreshSense_Results/yolov5s_food_spoilage/weights/best_pruned.pt'], imgsz=[640], batch_size=1, device=cpu, half=False, inplace=False, keras=False, optimize=False, int8=False, per_tensor=False, dynamic=False, cache=, simplify=True, mlmodel=False, opset=17, verbose=False, workspace=4, nms=False, agnostic_nms=False, topk_per_class=100, topk_all=100, iou_thres=0.45, conf_thres=0.25, include=['onnx']
YOLOv5 🚀 v7.0-472-g7ca403a3 Python-3.12.13 torch-2.10.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7034398 parameters, 0 gradients, 15.8 GFLOPs

PyTorch: starting from /content/drive/MyDrive/FreshSense_Results/yolov5s_food_spoilage/weights/best_pruned.pt with output shape (1, 25200, 14) (27.3 MB)

ONNX: starting export with onnx 1.21.0...
W0419 10:39:26.198000 28096 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requ